In [ ]:
import pickle
import csv
import datetime
import json
import torch
import faiss
import os
import math
import re
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from torch_geometric.data import Data
import itertools
from collections import Counter
import gensim
import numpy as np
import scipy.sparse as sp

In [ ]:
BASE_PATH = "Fakeddit"

### Loading the source posts

In [ ]:
df=pd.read_csv(os.path.join(BASE_PATH,"all_train.tsv"), sep='\t', header=0)
df2=pd.read_csv(os.path.join(BASE_PATH,"all_validate.tsv"), sep='\t', header=0)
df3=pd.read_csv(os.path.join(BASE_PATH,"all_test_public.tsv"), sep='\t', header=0)

In [ ]:
df.head()

In [ ]:
## Combining train,test and valid into one dataframe and clearing those don't have any text in it
combined_df=pd.DataFrame()
combined_df = pd.concat([df,df2,df3], ignore_index=True)
combined_df = combined_df.dropna(subset=["clean_title"])
print(len(combined_df))
combined_df = combined_df.reset_index(drop=True)

### Loading the Comments

In [ ]:
comments=pd.read_csv(os.path.join(BASE_PATH,"all_comments-Copy1.tsv"), sep='\t', header=0)

In [ ]:
comments.head()

### Mapping Source posts to comment chains

In [ ]:
top_post_ids=[i[3:] for i in comments["parent_id"] if str(i)!='nan' and i[0:3]=='t3_']
top_post_ids = list(set(top_post_ids))

In [ ]:
print(len(top_post_ids))

In [ ]:
mask = combined_df["id"].isin(top_post_ids)
all_top_posts = combined_df[mask]
print(len(all_top_posts))

In [ ]:
all_top_posts=all_top_posts[['id','created_utc','author','clean_title','2_way_label','num_comments']]

In [ ]:
labels_mapping = dict()
times_mapping = dict()
for i,j,k in zip(all_top_posts["id"],all_top_posts["2_way_label"],all_top_posts["created_utc"]):
    labels_mapping[i]=j
    dt_object = datetime.datetime.fromtimestamp(k)
    times_mapping[i]=dt_object.strftime("%Y-%m")
print(len(labels_mapping))

In [ ]:
uni=list(all_top_posts["id"].unique())
comm=comments[comments["submission_id"].isin(uni)]
comm.loc[:,'parent_id'] = comm['parent_id'].copy().str[3:]
grp=comm.groupby(["submission_id"])

### Loading the required files - User embeddings, sentence embeddings and id files

In [ ]:
with open("train_ids.pkl","rb") as f:
     train_ids = pickle.load(f)
with open("test_ids.pkl","rb") as f:
     test_ids = pickle.load(f)
with open("valid_ids.pkl","rb") as f:
     valid_ids = pickle.load(f)

In [ ]:
# Load user embeddings and mapping 
with open("User_features.pkl","rb") as f:
    user_vectors = pickle.load(f)

with open("user_to_id.json","r") as f:
    user_to_id = json.load(f)  


In [ ]:
with open("id_to_int.pkl","rb") as f:
     id_to_int = pickle.load(f)

with open("sentence_embeds_bert.pkl","rb") as f:
     sentence_embeddings = pickle.load(f)

### DP_TREE representation of comments

In [ ]:
l=0
for name,group in grp:
    l+=1
    my_dic = dict()
    my_second=dict()
    t=0
    my_dic[name[0]] = t
    my_second[t]=name[0]
    t+=1
    
    for i,sent in zip(group["id"],group["body"]):
        if str(sent)!='nan':
           my_dic[i]=t
           my_second[t]=i
           t+=1
    
    group['id'] = group['id'].map(my_dic)
    group['parent_id'] = group['parent_id'].map(my_dic)
    
        
    for id,p_id in zip(group['id'],group['parent_id']):
        
        if np.isnan(id)==False and np.isnan(p_id)==False and p_id>0:
            sentence_embeddings[id_to_int[my_second[id]]]+=sentence_embeddings[id_to_int[my_second[p_id]]]
    if(l%10000==0):
        print(l)
    # break


### Preparing train and validation groups

In [ ]:
mask = all_top_posts["id"].isin(train_ids)
train_posts = all_top_posts[mask]

In [ ]:
train_uni=list(train_posts["id"].unique())
print(len(train_uni))
train_comm=comments[comments["submission_id"].isin(train_uni)]
train_comm.loc[:,'parent_id'] = train_comm['parent_id'].copy().str[3:]
train_grp=train_comm.groupby(["submission_id"])

In [ ]:
valid_uni=list(valid_posts["id"].unique())
valid_comm=comments[comments["submission_id"].isin(valid_uni)]
valid_comm.loc[:,'parent_id'] = valid_comm['parent_id'].copy().str[3:]
valid_grp=valid_comm.groupby(["submission_id"])


## faiss indexing of source posts

In [ ]:
with open("train_user_list.pkl","rb") as f:
    all_train_user=pickle.load(f)

train_no_embed=set(list(set(all_train_user)-set(list(user_to_id.keys()))))
train_embed=set(list(user_to_id.keys()))

In [ ]:
id_to_user=dict()
for text,id_,user in zip(train_posts["clean_title"],train_posts["id"],train_posts["author"]):
    if(str(text)!="nan") and user not in train_no_embed:
        id_to_user[id_]=user

for na,gr in train_grp:
    for com,te,us in zip(gr["id"],gr["body"],gr["author"]):
        if str(te)!="nan" and str(us)!="nan" and us not in train_no_embed:
            id_to_user[com]=us

In [ ]:
post_indices=[]
for text,id_,user in zip(train_posts["clean_title"],train_posts["id"],train_posts["author"]):
        if str(user)!="nan" and user not in train_no_embed:
            post_indices.append(id_to_int[id_])

In [ ]:
post_sample=sentence_embeddings[post_indices]

In [ ]:
int_to_id={int:id for id,int in id_to_int.items()}

In [ ]:
int_to_id_post=dict()
t=0
for ind in post_indices:
    int_to_id_post[t]=int_to_id[ind]
    t+=1

post_sample_np=post_sample.numpy().copy()
post_index=faiss.IndexFlatIP(256)

faiss.normalize_L2(post_sample_np)
post_index.add(post_sample_np)

column_averages = np.mean(user_vectors, axis=0)
dummy_embed=column_averages


user_avg_embedding=dict()
post_comment_id_user=dict()

for name, group in valid_grp:
   
    id_user=dict()
    user_all_posts=dict()
    for text,id_,user in zip(group["body"],group["id"],group["author"]):
        if str(text)=="nan":
            continue
        user_all_posts[id_]=sentence_embeddings[id_to_int[id_]]
        id_user[id_]=user
    # post_all_users[name[0]]=user_all_posts    
    user_avg_embedding[name[0]]=user_all_posts
    post_comment_id_user[name[0]]=id_user

In [ ]:
all_train_post_comment=dict()
for na,gr in train_grp:          
        ind=[]
        f=0
        for com,te,us in zip(gr["id"],gr["body"],gr["author"]):
            if str(te)!="nan" and str(us)!="nan" and us not in train_no_embed:
                ind.append(id_to_int[com])
                f+=1
        
        all_train_post_comment[na[0]]=ind


all_post_matched=dict()
t=0
for text,id_,user in zip(valid_posts["clean_title"],valid_posts["id"],valid_posts["author"]):
        if(str(text)!="nan") :
            
            text_embedding=sentence_embeddings[id_to_int[id_]].numpy().copy()
            texter=text_embedding.reshape(1,256)
            faiss.normalize_L2(texter)
            # pattern = r'^post_\d+$' 
            k = 30
            scores,ann= post_index.search(texter, k=k)
            if(t%1000==0):
                print(t)
                
            t+=1
            all_post_matched[id_]=ann[0]

### Helping function to create dataset and load the model

In [ ]:
def get_node_features_with_user_and_bert(nodes_dic,sentences,id_to_int,valid_embed,comment_final_embed,post_id):
    num_nodes = len(nodes_dic)
    shape = (num_nodes,384)
    torch_tensor = torch.zeros(shape)
    # print(num_nodes)
    for i in nodes_dic:
        torch_tensor[nodes_dic[i]][:256] = sentences[id_to_int[i]]
        if i==post_id:
            torch_tensor[nodes_dic[i]][256:] = torch.tensor(valid_embed[i])
        else:
            
            torch_tensor[nodes_dic[i]][256:] = torch.tensor(comment_final_embed[post_id][i])
    if torch.isnan(torch_tensor).any():
        print("SOME ISSUE")
        
    return torch_tensor

In [ ]:
class MyOwnDataset(Dataset):
    def __init__(self, root, transform=None, pre_transform=None, pre_filter=None):
        super().__init__(root, transform, pre_transform, pre_filter)

    @property
    def raw_file_names(self):
        return ""

    @property
    def processed_file_names(self):
        files = os.listdir(self.processed_dir)
        filtered_files = [file for file in files if file.endswith(".pt")]
        return filtered_files

    def download(self):
        # Download to `self.raw_dir`.
        pass

    def process(self):
       
        for raw_path in self.raw_paths:
            # Read data from `raw_path`.
            data = Data(...)

            if self.pre_filter is not None and not self.pre_filter(data):
                continue

            if self.pre_transform is not None:
                data = self.pre_transform(data)

            torch.save(data, osp.join(self.processed_dir, f'data_{idx}.pt'))
            idx += 1

    def len(self):
        return len(self.processed_file_names)

    def get(self, idx):
        data = torch.load(osp.join(self.processed_dir, f'data_{idx}.pt'))
        return data

In [ ]:
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool
class GCN(torch.nn.Module):
    def __init__(self, hidden_channels,in_dim,out_dim):
        super(GCN, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(in_dim, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, out_dim)

    def forward(self, x, edge_index, batch, node_weight):
        # 1. Obtain node embeddings
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.conv3(x, edge_index)
       

        # 2. Readout layer
        #y = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        unique_batches = torch.unique(batch,sorted=True)
        final_weights = []
        for b in unique_batches:
            mask = (batch == b)
            batch_weights = batch[mask]
            for i,value in enumerate(batch[mask]):
                if i==0:
                    final_weights.append(node_weight)
                else:
                    final_weights.append((1-node_weight)/(len(batch_weights)-1))
                if len(batch_weights)==1:
                    print("Problem Here")

        final_weights = torch.tensor(final_weights)
        #print(final_weights)
        sum_weighted = torch.zeros(batch.max().item() + 1, x.size(1), device=x.device)
        sum_weighted.index_add_(0, batch, x * final_weights.view(-1, 1))

        x = sum_weighted  # [batch_size, hidden_channels]

        #print(x==y)
        # 3. Apply a final classifier
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)

        return x


In [ ]:
def test_analysis(loader,model_path,weight):
     model = torch.load(model_path)
     weight_param=weight
     model.eval()
     correct = 0
     actuals = []
     predictions = []
     for idx,data in enumerate(loader):  # Iterate in batches over the training/test dataset.

         # out = model(data.x, data.edge_index, data.batch)  
         out = model(data.x, data.edge_index, data.batch,weight_param)
         pred = out.argmax(dim=1)  # Use the class with highest probability.
         correct += int((pred == data.y).sum())  # Check against ground-truth labels.
         
         actuals.extend(np.array(data.y))
         predictions.extend(np.array(pred))
         
     return actuals,predictions

In [ ]:
weight_param = weight_obtained # Choose the lambda obtained from previous file
model_path = f"model_{weight_param}.pt"
best_acc=0.0000
best_k1=0
best_k2=0


directory=f'final_heuristics/dataset'
if not os.path.exists(f'{directory}/processed'):
    os.makedirs(f'{directory}/processed')
    
for k1 in range(10,31):
    valid_embed=dict()
    t=0
    valid_map_final=dict()
    for text,id_,user in zip(valid_posts["clean_title"],valid_posts["id"],valid_posts["author"]):
        if(str(text)!="nan") :
            emb=all_post_matched[id_][:k1]
            
            valid_map_final[id_]=emb##Dictionary to store the the most similar train post corresponding to each test post
            
            if user in train_embed:
                valid_embed[id_]=user_vectors[user_to_id[user]]
            else:
                all_similar=set(emb)
                embeddings=[user_vectors[user_to_id[id_to_user[int_to_id_post[ind]]]] for ind in all_similar]
                # print(len(embeddings))
                stacked_tensors = np.stack(embeddings, axis=0)
            
                # Calculate the average tensor
                average_tensor = np.mean(stacked_tensors, axis=0)
                valid_embed[id_]= average_tensor

            t+=1

    comment_constant_embed={}
    faiss_comments={}
    
    # Ensure 't' is initialized
    t = 0
    
    # Loop through the test groups
    for name, group in valid_grp:
        comment_constant_embed[name[0]] = {}
        faiss_comments[name[0]] = {}
        
        all_ids = set(list(user_avg_embedding[name[0]].keys()))
        train_posts = valid_map_final[name[0]]
        comments_ids = []
        id_user=post_comment_id_user[name[0]]
        for post in train_posts:
            comments_ids.extend(all_train_post_comment[int_to_id_post[post]])
          
        t += 1     
        if len(comments_ids) == 0:
            for id in all_ids:
                user=id_user[id]
                if user in train_embed:
                    comment_constant_embed[name[0]][id] = user_vectors[user_to_id[user]]
                else:
                    comment_constant_embed[name[0]][id] = dummy_embed
        else:
            my_map = {t: id for t, id in enumerate(comments_ids)}
            emb = sentence_embeddings[comments_ids].numpy().copy()
            my_index = faiss.IndexFlatIP(256)
            faiss.normalize_L2(emb)
            my_index.add(emb)
           
            for id in all_ids:
                user=id_user[id]
                # print(f"Processing user: {user}")
                if user in train_embed:
                    comment_constant_embed[name[0]][id] = user_vectors[user_to_id[user]]
                else:
                    text_embedding = user_avg_embedding[name[0]][id].numpy().copy()
                    texter = text_embedding.reshape(1, 256)
                    faiss.normalize_L2(texter)
                    p = min(len(emb), 200)
                    scores, ann = my_index.search(texter, k=p)
                    all_similar = ann[0]
                    faiss_comments[name[0]][id] = [int_to_id[my_map[ind]] for ind in all_similar]
                    

    for k2 in range(11,101):
        comment_final_embed={}

        t=0
        id_user=dict()
        for name, group in valid_grp:
            comment_final_embed[name[0]]={}
            all_ids=set(list(user_avg_embedding[name[0]].keys()))
            id_user=post_comment_id_user[name[0]]
            my_dict=comment_constant_embed[name[0]]
            
            for id in all_ids:
                user=id_user[id]
                if id in my_dict:
                    comment_final_embed[name[0]][id]=my_dict[id]
                else:
                    emb=faiss_comments[name[0]][id]
                    if(len(emb)>k2):
                        emb=emb[:k2]
                    emb=set(emb)
                    embeddings=[user_vectors[user_to_id[id_to_user[ind]]] for ind in emb]
                    # print(len(embeddings))
                    stacked_tensors = np.stack(embeddings, axis=0)
                
                    # Calculate the average tensor
                    average_tensor = np.mean(stacked_tensors, axis=0)
                    # print(len(average_tensor))
                    comment_final_embed[name[0]][id]=average_tensor
            
            t+=1
        l=0
        id_data_mapping = dict()
        data_list_valid =[]
        for name,group in valid_grp:
            my_dic = dict()
            t=0
            my_dic[name[0]] = t
            t+=1
            
            for i,sent in zip(group["id"],group["body"]):
                if str(sent)!='nan':
                   my_dic[i]=t
                   t+=1
        
            group['id'] = group['id'].map(my_dic)
            group['parent_id'] = group['parent_id'].map(my_dic)
            
            edges_list =[]
            for id,p_id in zip(group['id'],group['parent_id']):
                edges_list.append([p_id,id])
                edges_list.append([id,p_id])
                
            edge_index = torch.tensor(edges_list)
            # print(comment_final_embed[name[0]])
            data = Data(x=get_node_features_with_user_and_bert(my_dic,sentences,id_to_int,valid_embed,comment_final_embed,name[0]),  ## Change this function according to data
                        edge_index=edge_index.t().contiguous(),
                        y=torch.tensor([int(labels_mapping[name[0]])]))

            if  torch.isnan(data.edge_index).any()==False:
                data_list_valid.append(data)
                # torch.save(data,f'{directory}/data_{str(l)}.pt')
                id_data_mapping[name[0]]=l
                l+=1

        valid_loader = DataLoader(data_list_valid, batch_size=256, shuffle=False)
        actuals,predictions = test_analysis(valid_loader,model_path,weight_param)
        precision, recall, f1, support = precision_recall_fscore_support(actuals, predictions, labels=np.unique(actuals))

        acc=accuracy_score(actuals,predictions)

        if acc>best_acc:
            for i,data in enumerate(data_list_valid):
                torch.save(data,f'{directory}/processed/data_{str(i)}.pt')
            print(f'precision:{np.array(precision)},recall:{np.array(recall)},f1:{np.array(f1)},support:{np.array(support)}')
            best_acc=acc
            best_k1=k1
            best_k2=k2
            with open(f'{directory}/details.json',"w") as f:
                json.dump(
                 {
                     "k1": k1,
                     "k2": k2,
                     "precision": precision.tolist(),
                     "recall": recall.tolist(),
                     "f1": f1.tolist(),
                     "support": support.tolist(),
                     "acc": acc
                 },
                   f,
                    indent=4
                )
            
        
        print(k1,k2)

print(f'Best k values {best_k1},{best_k2}')
print(f'Best_acc {best_acc}')